# Cell 1: Install & Imports

In [1]:
# Install the correct libraries for the Kaggle Python 3.12 environment
# !pip install transformers==4.35.0 tf-keras 
!pip install transformers==4.39.3 tf-keras xgboost scikit-learn
import os
import warnings
warnings.filterwarnings('ignore')

# Force the current notebook kernel to use Legacy Keras (just in case)
os.environ["TF_USE_LEGACY_KERAS"] = "1"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 95.5 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the follow

Cell 2: Configuration (config.py)
This file controls your whole project. Change paths here only once.

In [2]:
%%writefile config.py
import os

# --- MASTER PATHS ---
BASE_DATA_PATH = "/kaggle/input/datasets/artechie001/cfdf-preprocessed-dataset-non-splited/CelebDF_dataset_split" 
TRAIN_PATH = os.path.join(BASE_DATA_PATH, "train")
VAL_PATH = os.path.join(BASE_DATA_PATH, "val")
TEST_PATH = os.path.join(BASE_DATA_PATH, "test")

# --- HYPERPARAMETERS ---
IMG_SIZE = (224, 224)
BATCH_SIZE_PER_REPLICA = 32 
EPOCHS = 20
LEARNING_RATE = 1e-4
DATA_SUBSET_RATIO = 0.99  

# --- MODEL WEIGHTS ---
VIT_WEIGHTS_FILE = "/kaggle/input/models/artechie001/celeb-df-models-final/tensorflow2/default/1/vit_best.weights.h5"
XCP_ATTN_WEIGHTS_FILE = "xcp_attn_best.weights.h5"
XCP_BASE_WEIGHTS_FILE = "xcp_base_best.weights.h5"
EFFB5_WEIGHTS_FILE = "/kaggle/input/models/artechie001/celeb-df-models-final/tensorflow2/default/1/effb5_best.weights.h5"
DENSE_WEIGHTS_FILE = "dense201_best.weights.h5"

# --- FEATURE EXTRACTION PATHS ---
FEATURE_DIR = "/kaggle/input/datasets/artechie001/cfdf-db5-vit-features/Features_cfdf"
MASK_FILE = "feature_mask.npy"
XGB_MODEL_FILE = "final_xgboost_model.pkl"

Writing config.py


# Cell 8: BGWO Optimizer (info_optimizer.py)
Your custom feature selection logic.

In [3]:
%%writefile bgwo_optimizer.py
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score

class BGWO_FeatureSelection:
    def __init__(self, num_agents=10, max_iter=5):
        self.n = num_agents
        self.max_iter = max_iter
        self.best_score = -1
        self.best_pos = None

    def fitness(self, pos, X, y):
        mask = pos > 0.5
        if np.sum(mask) == 0: mask[0] = True
        
        X_sub = X[:, mask]
        # Fast evaluation
        clf = XGBClassifier(n_estimators=30, max_depth=3, eval_metric='logloss')
        
        # 3-fold cross-validation
        scores = cross_val_score(clf, X_sub, y, cv=3)
        return scores.mean()

    def optimize(self, X, y):
        print(f"Optimizing {X.shape[1]} features using BGWO...")
        self.dim = X.shape[1]
        
        # Initialize pack positions randomly (0 or 1)
        self.X_pos = np.random.randint(2, size=(self.n, self.dim)).astype(float)
        
        Alpha_pos = np.zeros(self.dim)
        Alpha_score = -float("inf")
        
        Beta_pos = np.zeros(self.dim)
        Beta_score = -float("inf")
        
        Delta_pos = np.zeros(self.dim)
        Delta_score = -float("inf")
        
        for t in range(self.max_iter):
            # 1. Evaluate fitness and rank the pack hierarchy
            for i in range(self.n):
                score = self.fitness(self.X_pos[i], X, y)
                
                if score > Alpha_score:
                    Delta_score = Beta_score
                    Delta_pos = Beta_pos.copy()
                    Beta_score = Alpha_score
                    Beta_pos = Alpha_pos.copy()
                    Alpha_score = score
                    Alpha_pos = self.X_pos[i].copy()
                elif score > Beta_score:
                    Delta_score = Beta_score
                    Delta_pos = Beta_pos.copy()
                    Beta_score = score
                    Beta_pos = self.X_pos[i].copy()
                elif score > Delta_score:
                    Delta_score = score
                    Delta_pos = self.X_pos[i].copy()
                    
            print(f"Iter {t+1}: Best Acc (Alpha Wolf) {Alpha_score:.4f}")
            
            # 2. Update pack positions based on Alpha, Beta, Delta
            a = 2.0 - t * (2.0 / self.max_iter) # Decreases linearly from 2 to 0
            
            for i in range(self.n):
                for j in range(self.dim):
                    # Alpha's pull
                    r1 = np.random.rand()
                    r2 = np.random.rand()
                    A1 = 2.0 * a * r1 - a
                    C1 = 2.0 * r2
                    D_alpha = abs(C1 * Alpha_pos[j] - self.X_pos[i][j])
                    X1 = Alpha_pos[j] - A1 * D_alpha
                    
                    # Beta's pull
                    r1 = np.random.rand()
                    r2 = np.random.rand()
                    A2 = 2.0 * a * r1 - a
                    C2 = 2.0 * r2
                    D_beta = abs(C2 * Beta_pos[j] - self.X_pos[i][j])
                    X2 = Beta_pos[j] - A2 * D_beta
                    
                    # Delta's pull
                    r1 = np.random.rand()
                    r2 = np.random.rand()
                    A3 = 2.0 * a * r1 - a
                    C3 = 2.0 * r2
                    D_delta = abs(C3 * Delta_pos[j] - self.X_pos[i][j])
                    X3 = Delta_pos[j] - A3 * D_delta
                    
                    # Continuous step mathematical average
                    X_new = (X1 + X2 + X3) / 3.0
                    
                    # Clip to prevent overflow
                    X_new_clipped = np.clip(X_new, -10, 10)
                    
                    # Squashing function (Sigmoid) to convert movement to a probability
                    prob = 1.0 / (1.0 + np.exp(-X_new_clipped))
                    
                    # Binary flip: Keep (1) or Drop (0)
                    if np.random.rand() < prob:
                        self.X_pos[i][j] = 1.0
                    else:
                        self.X_pos[i][j] = 0.0

        self.best_score = Alpha_score
        self.best_pos = Alpha_pos > 0.5
        return self.best_pos

Writing bgwo_optimizer.py


# Cell 9: Train XGBoost (train_xgboost.py)
Trains the final classifier using the optimized features.

In [4]:
%%writefile train_xgboost.py
import config
import numpy as np
import joblib
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from bgwo_optimizer import BGWO_FeatureSelection

# 1. Load Data
print("Loading features...")
X_train = np.load(f"{config.FEATURE_DIR}/X_train.npy")
y_train = np.load(f"{config.FEATURE_DIR}/y_train.npy")
X_test = np.load(f"{config.FEATURE_DIR}/X_test.npy")
y_test = np.load(f"{config.FEATURE_DIR}/y_test.npy")

# 2. Optimize
subset_size = 5000 

print(f"Running BGWO Optimization on {subset_size} samples...")

if len(X_train) > subset_size:
    X_sub, _, y_sub, _ = train_test_split(
        X_train, y_train, 
        train_size=subset_size, 
        stratify=y_train,   
        random_state=42     
    )
else:
    X_sub = X_train
    y_sub = y_train

opt = BGWO_FeatureSelection(num_agents=10, max_iter=10)
mask = opt.optimize(X_sub, y_sub)

# Force at least some features to be selected if mask is empty
if np.sum(mask) == 0:
    print("WARNING: Optimizer selected 0 features. Reverting to all features.")
    mask[:] = True

np.save(config.MASK_FILE, mask)
print(f"Selected {np.sum(mask)} features out of {len(mask)}.")

# 3. Train Final Classifier (On FULL Training Data)
print("Training Final XGBoost on FULL data with BGWO selected features...")
model = XGBClassifier(
    n_estimators=1000,      
    learning_rate=0.05, 
    max_depth=7,            
    subsample=0.8,          
    colsample_bytree=0.8,
    tree_method='hist',     
    device='cuda'           
)

model.fit(X_train[:, mask], y_train)

# 4. Evaluate
print("Evaluating...")
preds = model.predict(X_test[:, mask])
print(classification_report(y_test, preds, target_names=['Real', 'Fake']))

joblib.dump(model, config.XGB_MODEL_FILE)

Writing train_xgboost.py


In [5]:
!python train_xgboost.py

Loading features...
Running BGWO Optimization on 5000 samples...
Optimizing 2816 features using BGWO...
Iter 1: Best Acc (Alpha Wolf) 0.9998
Iter 2: Best Acc (Alpha Wolf) 0.9998
Iter 3: Best Acc (Alpha Wolf) 0.9998
Iter 4: Best Acc (Alpha Wolf) 0.9998
Iter 5: Best Acc (Alpha Wolf) 0.9998
Iter 6: Best Acc (Alpha Wolf) 0.9998
Iter 7: Best Acc (Alpha Wolf) 0.9998
Iter 8: Best Acc (Alpha Wolf) 0.9998
Iter 9: Best Acc (Alpha Wolf) 0.9998
Iter 10: Best Acc (Alpha Wolf) 0.9998
Selected 1432 features out of 2816.
Training Final XGBoost on FULL data with BGWO selected features...
Evaluating...
/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [02:37:05] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device or

In [6]:
import numpy as np
import config

# 1. Load the mask generated by train_xgboost.py
mask = np.load(config.MASK_FILE)

# 2. Define the split point based on the ViT + EfficientNetB5 hybrid architecture
vit_dim = 768
eff_dim = 2048

# 3. Slice the mask to see which backbone contributed more
vit_mask = mask[:vit_dim]
eff_mask = mask[vit_dim:]

# 4. Calculate counts
vit_selected = np.sum(vit_mask)
eff_selected = np.sum(eff_mask)
total_selected = np.sum(mask)

# 5. Print the accurate breakdown for the research paper
print(f"--- FEATURE SELECTION BREAKDOWN ---")
print(f"Total Features before INFO: 2816")
print(f"Total Features after INFO:  {total_selected} ({total_selected/2816*100:.2f}%)")
print("-" * 40)
print(f"ViT Features Selected:            {vit_selected} out of {vit_dim}")
print(f"EfficientNetB5 Features Selected: {eff_selected} out of {eff_dim}")
print("-" * 40)

# 6. Calculate Feature Importance Retention
vit_ratio = (vit_selected / vit_dim) * 100
eff_ratio = (eff_selected / eff_dim) * 100

print(f"ViT Retention Rate:            {vit_ratio:.2f}%")
print(f"EfficientNetB5 Retention Rate: {eff_ratio:.2f}%")

--- FEATURE SELECTION BREAKDOWN ---
Total Features before INFO: 2816
Total Features after INFO:  1432 (50.85%)
----------------------------------------
ViT Features Selected:            386 out of 768
EfficientNetB5 Features Selected: 1046 out of 2048
----------------------------------------
ViT Retention Rate:            50.26%
EfficientNetB5 Retention Rate: 51.07%


In [7]:
%%writefile test_final_system.py
import config
import numpy as np
import joblib
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, precision_score, recall_score, f1_score

def test_optimized_system():
    print("--- FINAL EVALUATION: ViT + EfficientNetB5 + BGWO + XGBoost ---")
    
    print("Loading Test Features...")
    try:
        X_test = np.load(f"{config.FEATURE_DIR}/X_test.npy")
        y_test = np.load(f"{config.FEATURE_DIR}/y_test.npy")
    except FileNotFoundError:
        print("ERROR: Feature files not found. Did you run your extraction script?")
        return

    print("Loading Optimized Model & Mask...")
    try:
        mask = np.load(config.MASK_FILE)
        model = joblib.load(config.XGB_MODEL_FILE)
    except FileNotFoundError:
        print("ERROR: Model/Mask not found. Did you run 'train_xgboost.py'?")
        return
    
    # Filter to only the features selected by the optimizer
    X_test_opt = X_test[:, mask]
    
    print(f"\nTesting on {len(y_test)} samples using {X_test_opt.shape[1]} optimized features...\n")
    
    # Extract both class predictions and probabilities
    preds = model.predict(X_test_opt)
    probs = model.predict_proba(X_test_opt)[:, 1] 
    
    print("="*50)
    print("   OPTIMIZED RESULTS (ViT + EffB5 + BGWO + XGB)")
    print("="*50)
    
    # Calculate comprehensive metrics
    acc = accuracy_score(y_test, preds)
    auc = roc_auc_score(y_test, probs)
    prec = precision_score(y_test, preds)
    rec = recall_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    
    print(f"Accuracy : {acc*100:.2f}%")
    print(f"AUC      : {auc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-Score : {f1:.4f}")
    print("-" * 50)
    print(classification_report(y_test, preds, target_names=['Real', 'Fake']))

if __name__ == "__main__":
    test_optimized_system()

Writing test_final_system.py


In [8]:
!python test_final_system.py

--- FINAL EVALUATION: ViT + EfficientNetB5 + BGWO + XGBoost ---
Loading Test Features...
Loading Optimized Model & Mask...

Testing on 3346 samples using 1432 optimized features...

/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [02:37:08] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
   OPTIMIZED RESULTS (ViT + EffB5 + BGWO + XGB)
Accuracy : 98.80%
AUC      : 0.9996
Precision: 0.9828
Recall   : 0.9934
F1-Score : 0.9881
--------------------------------------------------
              precision    recall  f1-score   support

        Real       

In [9]:
%%writefile utils.py
import config
import os
import glob
import tensorflow as tf
from sklearn.utils import shuffle

def get_image_paths(data_path, subset_ratio=config.DATA_SUBSET_RATIO):
    """
    Retrieves and shuffles image paths from the nested directory structure.
    """
    fake_paths = glob.glob(os.path.join(data_path, 'fake', '*', '*.*'))
    real_paths = glob.glob(os.path.join(data_path, 'real', '*', '*.*'))
    
    paths = fake_paths + real_paths
    labels = [1] * len(fake_paths) + [0] * len(real_paths) # 1 for Fake, 0 for Real
    
    paths, labels = shuffle(paths, labels, random_state=42)
    
    if subset_ratio < 1.0:
        limit = int(len(paths) * subset_ratio)
        paths = paths[:limit]
        labels = labels[:limit]
        
    print(f"Loaded {len(paths)} images from {data_path}")
    return paths, labels

def load_and_preprocess_image(path, label):
    """
    Native TensorFlow function to read, decode, resize, and normalize images.
    Crucial for preventing CPU bottlenecks during multi-GPU training.
    """

    img = tf.io.read_file(path)

    img = tf.image.decode_image(img, channels=3, expand_animations=False)

    img = tf.image.resize(img, config.IMG_SIZE)

    img = tf.cast(img, tf.float32) / 255.0
    

    label = tf.one_hot(label, depth=2)
    
    return img, label

def create_tf_dataset(paths, labels, batch_size, is_training=True):
    """
    Builds a highly optimized tf.data.Dataset pipeline.
    Uses AUTOTUNE to dynamically allocate CPU threads for data loading.
    """

    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    
    if is_training:

        dataset = dataset.shuffle(buffer_size=len(paths), reshuffle_each_iteration=True)
        

    dataset = dataset.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
    

    dataset = dataset.batch(batch_size)
    

    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    
    return dataset

Writing utils.py


In [10]:
%%writefile models.py
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tensorflow as tf
from tensorflow.keras import layers, models, Model
from tensorflow.keras.applications import Xception, EfficientNetB5, DenseNet201
from transformers import TFViTModel
from tensorflow.keras import mixed_precision

# Ensure final layers output in float32 for mixed precision stability
FINAL_DTYPE = 'float32'

class ViTWrapper(layers.Layer):
    """Custom layer to wrap the HuggingFace ViT model natively into Keras."""
    def __init__(self, vit_model, **kwargs):
        super().__init__(**kwargs)
        self.vit_model = vit_model
        
    def call(self, inputs):
        x = tf.transpose(inputs, perm=[0, 3, 1, 2])
        outputs = self.vit_model.vit(pixel_values=x)
        return outputs.last_hidden_state[:, 0, :]
        
    def get_config(self):
        return super().get_config()

class AttentionLayer(layers.Layer):
    """Custom Attention Mechanism for feature weighting."""
    def __init__(self, dim, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)
        self.dim = dim
        
    def get_config(self):
        config = super().get_config()
        config.update({"dim": self.dim})
        return config
        
    def build(self, input_shape):
        self.dense = layers.Dense(self.dim, activation='tanh', use_bias=True)
        self.u_vec = self.add_weight(name='u_vec', shape=(self.dim, 1), initializer='uniform', trainable=True)
        super(AttentionLayer, self).build(input_shape)
        
    def call(self, x):
        u_it = self.dense(x)
        score = tf.matmul(u_it, self.u_vec)
        weights = tf.nn.softmax(score, axis=1)
        return tf.reduce_sum(x * weights, axis=1)

# 1. ViT
def build_vit_classifier(input_shape=(224, 224, 3)):
    inputs = layers.Input(shape=input_shape)
    norm_layer = layers.Normalization(mean=[0.485, 0.456, 0.406], variance=[0.229**2, 0.224**2, 0.225**2])
    x = norm_layer(inputs)
    try:
        vit_model = TFViTModel.from_pretrained('google/vit-base-patch16-224', from_pt=True)
    except:
        vit_model = TFViTModel.from_pretrained('google/vit-base-patch16-224')
    vit_model.trainable = True
    features = ViTWrapper(vit_model, name='vit_features')(x)
    x = layers.Dense(512, activation='relu')(features)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(2, activation='softmax', dtype=FINAL_DTYPE)(x)
    return Model(inputs, outputs, name="ViT_Classifier")

# 2. Xception + Custom Attention
def build_xception_attn_classifier(input_shape=(224, 224, 3)):
    inputs = layers.Input(shape=input_shape)
    x = layers.Rescaling(scale=2.0, offset=-1.0)(inputs) 
    x = layers.GaussianNoise(0.05)(x)
    base = Xception(include_top=False, weights='imagenet', input_tensor=x)
    base.trainable = True
    x = base.output
    x = layers.Reshape((x.shape[1]*x.shape[2], x.shape[3]))(x)
    features = AttentionLayer(dim=512, name='xcp_features')(x)
    x = layers.Dense(512, activation='relu')(features)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(2, activation='softmax', dtype=FINAL_DTYPE)(x)
    return Model(inputs, outputs, name="Xception_Attn_Classifier")

# 3. Base Xception
def build_xception_base_classifier(input_shape=(224, 224, 3)):
    inputs = layers.Input(shape=input_shape)
    x = layers.Rescaling(scale=2.0, offset=-1.0)(inputs) 
    base = Xception(include_top=False, weights='imagenet', input_tensor=x, pooling='avg')
    base.trainable = True
    x = base.output
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(2, activation='softmax', dtype=FINAL_DTYPE)(x)
    return Model(inputs, outputs, name="Xception_Base_Classifier")

# 4. EfficientNetB5
def build_effb5_classifier(input_shape=(224, 224, 3)):
    inputs = layers.Input(shape=input_shape)
    x = layers.Rescaling(scale=255.0)(inputs) 
    
    # --- BULLETPROOF WORKAROUND FOR EFFICIENTNET ---
    # Temporarily drop back to float32 to bypass the internal hardcoded constants bug
    current_policy = mixed_precision.global_policy()
    mixed_precision.set_global_policy('float32')
    
    base = EfficientNetB5(include_top=False, weights='imagenet', pooling='avg')
    
    # Immediately restore the mixed precision policy
    mixed_precision.set_global_policy(current_policy)
    # -----------------------------------------------
    
    # Explicitly force input to float32 before feeding it to the base model
    x = tf.cast(x, tf.float32)
    x = base(x)
    
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(2, activation='softmax', dtype=FINAL_DTYPE)(x)
    return Model(inputs, outputs, name="EfficientNetB5_Classifier")

# 5. DenseNet201
def build_dense201_classifier(input_shape=(224, 224, 3)):
    inputs = layers.Input(shape=input_shape)
    x = layers.Rescaling(scale=255.0)(inputs) 
    
    # Apply standard preprocessing
    x = tf.cast(x, tf.float32)
    x = tf.keras.applications.densenet.preprocess_input(x) 
    
    # Apply the same safety wrapper for DenseNet
    current_policy = mixed_precision.global_policy()
    mixed_precision.set_global_policy('float32')
    
    base = DenseNet201(include_top=False, weights='imagenet', pooling='avg')
    
    mixed_precision.set_global_policy(current_policy)
    
    x = base(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(2, activation='softmax', dtype=FINAL_DTYPE)(x)
    return Model(inputs, outputs, name="DenseNet201_Classifier")

Writing models.py


In [11]:
%%writefile evaluate_test_set.py
import config
import os
import argparse
import numpy as np
import pandas as pd
import joblib
import cv2
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, roc_auc_score, roc_curve, precision_score, recall_score, f1_score
from tqdm import tqdm
from models import build_vit_classifier, build_effb5_classifier
from utils import get_image_paths

# Local preprocessor added here to avoid modifying the highly optimized utils.py
def preprocess_image(path):
    """
    Reads, resizes, and normalizes a single image array for the manual evaluation loop.
    """
    img = cv2.imread(path)
    if img is None: return None
    img = cv2.resize(img, config.IMG_SIZE)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = img.astype(np.float32) / 255.0
    return img

def evaluate_entire_test_set(custom_path=None):
    print("\n" + "="*50)
    print("      STARTING FULL HYBRID EVALUATION")
    print("="*50)
    
    # 1. Determine the test path
    target_path = custom_path if custom_path else config.TEST_PATH
    print(f" Target Directory: {target_path}")

    # 2. Load the Feature Extractors (Separated)
    print("1. Loading Deep Feature Extractors...")
    try:
        vit_model = build_vit_classifier()
        vit_model.load_weights(config.VIT_WEIGHTS_FILE)
        vit_extractor = tf.keras.Model(inputs=vit_model.input, outputs=vit_model.get_layer('vit_features').output)

        eff_model = build_effb5_classifier()
        eff_model.load_weights(config.EFFB5_WEIGHTS_FILE)
        # Bypass Keras Graph Disconnected bug by targeting the input of the 3rd to last layer
        eff_extractor = tf.keras.Model(inputs=eff_model.input, outputs=eff_model.layers[-3].input)
    except Exception as e:
        print(f" Error loading Deep Learning weights: {e}")
        return

    # 3. Load the ML Artifacts
    print("2. Loading XGBoost & Feature Mask...")
    try:
        mask = np.load(config.MASK_FILE)
        xgb_model = joblib.load(config.XGB_MODEL_FILE)
        print(f" Loaded Mask (Selected {np.sum(mask)} features)")
    except Exception as e:
        print(f" Error loading ML artifacts: {e}. Run train_xgboost.py first.")
        return

    # 4. Scan Test Images
    test_paths, test_labels = get_image_paths(target_path, subset_ratio=1.0)
    if len(test_paths) == 0:
        print(f" No images found in {target_path}!")
        return

    # 5. Batch Prediction Loop
    results = []
    batch_size = 32
    
    print(f"3. Processing {len(test_paths)} images...")
    for i in tqdm(range(0, len(test_paths), batch_size)):
        batch_paths = test_paths[i : i + batch_size]
        batch_lbls = test_labels[i : i + batch_size]
        
        batch_imgs = []
        valid_indices = []
        for idx, path in enumerate(batch_paths):
            img = preprocess_image(path)
            if img is not None:
                batch_imgs.append(img)
                valid_indices.append(idx)
        
        if not batch_imgs: continue
        batch_imgs_np = np.array(batch_imgs)
        
        # A. Extract Separate Features
        feat_vit = vit_extractor.predict(batch_imgs_np, verbose=0)
        feat_eff = eff_extractor.predict(batch_imgs_np, verbose=0)
        
        # B. Concatenate (Manual Fusion)
        deep_features = np.concatenate([feat_vit, feat_eff], axis=1)
        
        # C. Apply Mask & Predict
        selected_features = deep_features[:, mask]
        probs = xgb_model.predict_proba(selected_features) # [Real_Prob, Fake_Prob]
        preds = np.argmax(probs, axis=1)
        
        for j, v_idx in enumerate(valid_indices):
            results.append({
                "true": batch_lbls[v_idx],
                "pred": preds[j],
                "prob_fake": probs[j][1]
            })

    # 6. Analysis & Metrics
    df = pd.DataFrame(results)
    y_true = df['true'].values
    y_pred = df['pred'].values
    y_probs = df['prob_fake'].values

    acc = accuracy_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_probs)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)

    print("\n" + "="*50)
    print("           BENCHMARK REPORT")
    print("="*50)
    print(classification_report(y_true, y_pred, target_names=['Real', 'Fake']))
    print("-" * 50)
    print(f"Accuracy : {acc*100:.2f}%")
    print(f"AUC      : {auc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-Score : {f1:.4f}")
    print("="*50)

    # 7. Plotting Confusion Matrix
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
    plt.title('Confusion Matrix: Hybrid ViT-EffB5-XGB')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.savefig('confusion_matrix.png')
    print(" Confusion Matrix saved as 'confusion_matrix.png'")

    # 8. Plotting ROC Curve
    fpr, tpr, _ = roc_curve(y_true, y_probs)
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {auc:.4f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic (ROC)')
    plt.legend(loc="lower right")
    plt.savefig('roc_curve.png')
    print(" ROC Curve saved as 'roc_curve.png'")

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument('--path', type=str, default=None)
    args = parser.parse_args()
    
    # Force Keras 2 for evaluation to match training environment
    os.environ["TF_USE_LEGACY_KERAS"] = "1"
    
    evaluate_entire_test_set(custom_path=args.path)

Writing evaluate_test_set.py


In [12]:
!python evaluate_test_set.py --path "/kaggle/input/datasets/artechie001/faceforensics-pngs/FFPP_Splitted_Dataset/test"

2026-05-18 02:37:11.024756: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779071831.180484      94 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779071831.225152      94 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779071831.600435      94 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779071831.600471      94 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779071831.600475      94 computation_placer.cc:177] computation placer alr